# HTTP로 LLM 호출 — 강사용 정답본

원본: `src/solution/Chapter01/01-llm-api/01_requests.py`  
대응 슬라이드: `docs/raw/Chapter01/day01/1-2-development-environment-and-llm-api-mechanism.pdf`

**학습 목표:** 환경변수 → 인증 헤더 → POST → 상태 검사 → JSON 답변의 각 단계를 원본 코드에서 찾아 설명한다.

**실행 전:** 설정된 LLM 서버로 HTTP POST 요청 1회를 보낸다. 모델·인증·네트워크를 확인하고 키와 헤더 값을 출력하지 않는다.



## 흐름을 먼저 읽기

환경변수 → 인증 헤더 → POST → 상태 검사 → JSON 답변

핵심은 요청과 응답의 필드 역할이다. 입력이 어디서 들어오고 결과가 어디로 이동하는지 화살표를 따라가자. 원본 주석과 TODO를 그대로 둔다. TODO는 CodeMate에게 작업을 지시하고 결과를 검토할 지점으로 읽는다.


## CodeMate 실습 지시

> 다음 코드 흐름을 순서대로 설명해 주세요: 환경변수 → 인증 헤더 → POST → 상태 검사 → JSON 답변. raise_for_status를 JSON 파싱보다 먼저 호출하는 이유는? 원본 코드를 바꾸기 전에 근거가 되는 위치와 실행 위험을 알려 주세요.

CodeMate의 제안을 원본의 입력·출력·오류 흐름과 비교하고, 다른 결과가 나오면 다시 지시한다.


## 원본 코드 — 수동 실행

아래 셀은 `.py` 전체를 그대로 보존한다. 위에서 아래로 한 번에 실행되므로 환경과 부작용을 확인한다. **자동 실행/Run All을 사용하지 않는다.**

**이 코드셀의 실행 조건·부작용:** 설정된 LLM 서버로 HTTP POST 요청 1회를 보낸다. 모델·인증·네트워크를 확인하고 키와 헤더 값을 출력하지 않는다.



In [ ]:
import os
import uuid
from pprint import pprint

import requests
from dotenv import load_dotenv

load_dotenv()

provider = os.getenv('LLM_PROVIDER', 'samsung').strip().lower()
if provider == 'openai':
    api_base_url = 'https://api.openai.com/v1/chat/completions'
    model = os.getenv('OPENAI_MODEL')
    api_key = os.getenv('OPENAI_API_KEY')
    if not model or not api_key:
        raise RuntimeError('OPENAI_MODEL과 OPENAI_API_KEY가 필요합니다.')
    headers = {
        'Authorization': f'Bearer {api_key}',
        'Content-Type': 'application/json',
    }
elif provider == 'samsung':
    required = ('api_base_url', 'model', 'credential_key', 'send_system_name', 'user_id')
    missing = [name for name in required if not os.getenv(name)]
    if missing:
        raise RuntimeError(f'사내 LLM 설정이 부족합니다: {missing}')
    api_base_url = f'{os.getenv("api_base_url").rstrip("/")}/chat/completions'
    model = os.getenv('model')
    headers = {
        'x-dep-ticket': os.getenv('credential_key'),
        'Send-System-Name': os.getenv('send_system_name'),
        'User-Id': os.getenv('user_id'),
        'User-Type': 'AD_ID',
        'Prompt-Msg-Id': str(uuid.uuid4()),
        'Completion-Msg-Id': str(uuid.uuid4()),
        'Content-Type': 'application/json',
    }
else:
    raise ValueError(f'지원하지 않는 LLM_PROVIDER: {provider}')

payload = {
    "model": model,
    "messages": [
        { "role": "user","content": "LLM이 무엇인지 3문장으로 설명해줘." },
    ],
    "temperature": 0.5,
    "stream": False,
}
if provider == 'samsung':
    payload['include_reasoning'] = False
elif model == 'gpt-6-luna':
    # GPT-6 Luna의 Chat Completions는 기본 추론 설정에서 temperature를 받지 않는다.
    payload.pop('temperature')
    payload['reasoning_effort'] = 'none'

new_response = requests.post(
    api_base_url, headers=headers, json=payload, timeout=30
)
new_response.raise_for_status()
recv_json = new_response.json()
  
print("### 응답 Json")
pprint(recv_json)
print("### 결과 메시지")
print(recv_json["choices"][0]["message"]["content"])


"""
api_key = os.getenv('OPENAI_API_KEY')
model = os.getenv('OPENAI_MODEL', 'gpt-4o-mini')

if not api_key:
    raise RuntimeError('OPENAI_API_KEY 환경변수를 설정해주세요.')

# OpenAI Responses API 엔드포인트
# - 기존 Chat Completions(/v1/chat/completions)의 후속 API
# - input 필드 하나로 단순 문자열과 대화 내역 모두 처리 가능
url = 'https://api.openai.com/v1/responses'
headers = {
    'Authorization': f'Bearer {api_key}',
    'Content-Type': 'application/json',
}

payload = {
    'model': model,
    'input': 'LLM이 무엇인지 3문장으로 설명해줘.',
}

response = requests.post(url, headers=headers, json=payload, timeout=30)
response.raise_for_status()

data = response.json()
pprint(data)
"""


## 결과를 검토하기

- 예상 동작: 성공하면 HTTP 응답 JSON과 답변 문장이 출력된다.
- 확인 질문: raise_for_status를 JSON 파싱보다 먼저 호출하는 이유는?
- 오류가 나면 먼저 원본 코드의 어느 단계인지 찾고, CodeMate가 제안한 수정의 이유와 부작용을 설명하게 한다.
